# Laboratorio IA — Clasificación Multiclase con Regresión Logística
## Dataset: Animals-10 — Reconocimiento de Animales (Dataset Multimodal / Imagen)

**Estudiante:** Jorge Gabriel Escobar Moscoso  
**Docente:** Ariel Guarachi — 63706640

---

### Descripción del Dataset

**Animals-10** contiene imágenes de 10 categorías de animales recopiladas de Google Images. Las categorías son: perro (*cane*), gato (*gatto*), caballo (*cavallo*), araña (*ragno*), mariposa (*farfalla*), pollo (*gallina*), oveja (*pecora*), vaca (*mucca*), ardilla (*scoiattolo*) y elefante (*elefante*).

Para aplicar regresión logística, cada imagen se **aplana** en un vector de píxeles RGB redimensionado a 64×64 px, generando **12 288 features** por imagen.

| Característica | Valor |
|---|---|
| Número de features (n) | 12 288 (64×64×3 píxeles) |
| Número de ejemplos (m) | ≈ 26 000 imágenes |
| Tipo | Multimodal — imágenes RGB |
| Tarea | Clasificación multiclase (10 clases) |

Cumple los requisitos: **n > 100** y **m ≥ 5 000**.

> **Fuente:** https://www.kaggle.com/datasets/alessiocorrado99/animals10

---

### Estrategia general

Cada imagen RGB de tamaño arbitrario se redimensiona a 64×64 píxeles y se aplana a un vector de 12 288 valores. Se aplica normalización dividiendo entre 255 para llevar los píxeles al rango [0,1]. El modelo de regresión logística **One-vs-Rest (OvR)** entrena un clasificador binario por cada una de las 10 clases.

División: **80% entrenamiento / 20% prueba**.

### 1. Montar Google Drive y cargar dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
from PIL import Image
%matplotlib inline

### 2. Carga del Dataset

El dataset se organiza en carpetas: una carpeta por clase, con las imágenes dentro. Se recorre la estructura con `os.walk` y se carga cada imagen con PIL (Pillow), redimensionándola a 64×64 píxeles para homogeneizar las dimensiones.

La estructura esperada en Google Drive es:
```
Datasets/animals10/raw-img/
    cane/        ← perro
    gatto/       ← gato
    cavallo/     ← caballo
    ...
```

> **Nota:** El dataset original usa nombres en italiano. El mapeo a español se define en `NOMBRE_CLASES`.

In [ ]:
# ─── AJUSTAR ESTA RUTA AL DATASET EN TU DRIVE ────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/Datasets/animals10/raw-img'
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE   = 64          # redimensionar a 64x64 píxeles
MAX_POR_CLASE = 1000     # limitar por clase para balanceo y velocidad

# Mapeo carpeta → nombre de clase legible
NOMBRE_CLASES = {
    'cane':       'Perro',
    'gatto':      'Gato',
    'cavallo':    'Caballo',
    'ragno':      'Araña',
    'farfalla':   'Mariposa',
    'gallina':    'Pollo',
    'pecora':     'Oveja',
    'mucca':      'Vaca',
    'scoiattolo': 'Ardilla',
    'elefante':   'Elefante'
}

# Ordenar clases para índices reproducibles
CARPETAS_CLASES = sorted(NOMBRE_CLASES.keys())
ETIQUETAS       = [NOMBRE_CLASES[c] for c in CARPETAS_CLASES]
print('Clases (índice → nombre):')
for i, (fol, nom) in enumerate(zip(CARPETAS_CLASES, ETIQUETAS)):
    print(f'  {i}: {nom} ({fol})')

In [ ]:
def cargar_imagenes(data_dir, carpetas, img_size=64, max_por_clase=1000):
    """
    Carga imágenes desde subdirectorios, las redimensiona y aplana.
    Retorna X (m x n), y (m,) y un registro de la cantidad cargada por clase.
    """
    X_list = []
    y_list = []
    conteo = {}
    n_features = img_size * img_size * 3  # píxeles RGB aplanados

    for idx, carpeta in enumerate(carpetas):
        ruta = os.path.join(data_dir, carpeta)
        if not os.path.isdir(ruta):
            print(f'  [ADVERTENCIA] No se encontró la carpeta: {ruta}')
            continue

        archivos = [f for f in os.listdir(ruta)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        archivos = archivos[:max_por_clase]  # limitar para balanceo

        cargadas = 0
        for nombre_archivo in archivos:
            try:
                img = Image.open(os.path.join(ruta, nombre_archivo)).convert('RGB')
                img = img.resize((img_size, img_size))
                vector = np.array(img, dtype=np.float32).flatten() / 255.0
                X_list.append(vector)
                y_list.append(idx)
                cargadas += 1
            except Exception:
                pass  # ignorar imágenes corruptas

        conteo[carpeta] = cargadas

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=int)
    return X, y, conteo

print('Cargando imágenes... (puede tardar varios minutos en Drive)')
X_raw, y_raw, conteo = cargar_imagenes(DATA_DIR, CARPETAS_CLASES,
                                        img_size=IMG_SIZE,
                                        max_por_clase=MAX_POR_CLASE)

print(f'\nShape X: {X_raw.shape}  →  m={X_raw.shape[0]} ejemplos, n={X_raw.shape[1]} features')
print('Imágenes cargadas por clase:')
for fol, cnt in conteo.items():
    print(f'  {NOMBRE_CLASES[fol]:10s}: {cnt}')

### 3. Preprocesamiento con Pandas

Se construye un DataFrame de Pandas para aprovechar sus herramientas de inspección y balanceo. Cada fila es un ejemplo aplanado; la columna `clase` contiene el índice numérico de la categoría.

In [ ]:
# Construir DataFrame de Pandas con los datos cargados
col_names = [f'pixel_{i}' for i in range(X_raw.shape[1])]
df = pd.DataFrame(X_raw, columns=col_names)
df['clase'] = y_raw

print(f'DataFrame shape: {df.shape}')
print('Distribución de clases:')
print(df['clase'].value_counts().sort_index().rename(index={i: ETIQUETAS[i] for i in range(len(ETIQUETAS))}))

#### 3.1 Balanceo de clases

Se aplica submuestreo para igualar el número de ejemplos por clase, garantizando que el modelo no favorezca ninguna categoría durante el entrenamiento.

In [ ]:
min_count = df['clase'].value_counts().min()
print(f'Ejemplos por clase tras balanceo: {min_count}')

grupos = [df[df['clase'] == k].sample(n=min_count, random_state=42)
          for k in range(len(ETIQUETAS))]
df_bal = pd.concat(grupos).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Shape balanceado: {df_bal.shape}')

# Gráfica de distribución balanceada
counts_bal = df_bal['clase'].value_counts().sort_index()
pyplot.figure(figsize=(10, 4))
pyplot.bar(ETIQUETAS, counts_bal.values, color='darkorange', edgecolor='black')
pyplot.xlabel('Categoría de Animal')
pyplot.ylabel('Número de Imágenes')
pyplot.title('Distribución balanceada — Animals-10')
pyplot.xticks(rotation=30, ha='right')
pyplot.tight_layout()
pyplot.show()

#### 3.2 Visualización de ejemplos del dataset

Se muestran imágenes representativas de cada clase para verificar la carga correcta del dataset. Esto es especialmente importante en datasets de imágenes para detectar imágenes corruptas o mal etiquetadas.

In [ ]:
fig, axes = pyplot.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

X_bal = df_bal[[c for c in df_bal.columns if c != 'clase']].values
y_bal = df_bal['clase'].values

for k in range(len(ETIQUETAS)):
    idx = np.where(y_bal == k)[0][0]
    img = X_bal[idx].reshape(IMG_SIZE, IMG_SIZE, 3)
    axes[k].imshow(img)
    axes[k].set_title(ETIQUETAS[k], fontsize=10)
    axes[k].axis('off')

pyplot.suptitle('Ejemplos representativos por clase — Animals-10', fontsize=13, fontweight='bold')
pyplot.tight_layout()
pyplot.show()

#### 3.3 División 80/20 — Entrenamiento y Prueba

Los datos se dividen: 80% para entrenamiento y 20% para prueba. El conjunto de prueba **no participa en ningún paso del entrenamiento**.

Los píxeles ya están normalizados en [0,1] por la división entre 255 durante la carga. Se agrega el término de intercepción (columna de unos) a ambos conjuntos.

In [ ]:
m_total = X_bal.shape[0]
m_train = int(0.8 * m_total)

X_train = X_bal[:m_train]
y_train = y_bal[:m_train]
X_test  = X_bal[m_train:]
y_test  = y_bal[m_train:]

print(f'Total ejemplos     : {m_total}')
print(f'Entrenamiento (80%): {X_train.shape[0]}')
print(f'Prueba        (20%): {X_test.shape[0]}')

# Agregar término de intercepción
X_train_b = np.concatenate([np.ones((X_train.shape[0], 1)), X_train], axis=1)
X_test_b  = np.concatenate([np.ones((X_test.shape[0],  1)), X_test],  axis=1)

print(f'Shape X_train con bias: {X_train_b.shape}')
print(f'Shape X_test  con bias: {X_test_b.shape}')

### 4. Implementación del Modelo — Regresión Logística Multiclase (OvR)

#### 4.1 Función Sigmoidea

La función sigmoidea transforma cualquier valor real en una probabilidad entre 0 y 1:

$$g(z) = \frac{1}{1 + e^{-z}}$$

Para el caso de clasificación de imágenes, $z = \theta^T x$ donde $x$ es el vector de píxeles del ejemplo.

In [ ]:
def sigmoid(z):
    """Función sigmoidea — opera sobre escalares, vectores y matrices."""
    z = np.array(z, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-z))

# Verificación
print(f'sigmoid(0)    = {sigmoid(0)}   (esperado: 0.5)')
print(f'sigmoid(10)   = {sigmoid(10):.6f} (esperado: ~1.0)')
print(f'sigmoid(-10)  = {sigmoid(-10):.6f} (esperado: ~0.0)')

#### 4.2 Función de Costo y Gradiente

Se usa la entropía cruzada binaria como función de costo para cada clasificador OvR:

$$J(\theta) = \frac{1}{m} \sum_{i=1}^{m} \left[ -y^{(i)} \log h_\theta(x^{(i)}) - (1-y^{(i)}) \log(1-h_\theta(x^{(i)})) \right]$$

El gradiente vectorizado es:

$$\nabla J(\theta) = \frac{1}{m} X^T (h_\theta(X) - y)$$

In [ ]:
def costFunction(theta, X, y):
    """
    Costo J y gradiente para regresión logística binaria.
    theta: (n+1,), X: (m x n+1), y: (m,) binario
    Devuelve (J, grad).
    """
    m = y.size
    h = sigmoid(X.dot(theta))
    eps = 1e-12
    J = (1/m) * np.sum(-y * np.log(h + eps) - (1-y) * np.log(1 - h + eps))
    grad = (1/m) * X.T.dot(h - y)
    return J, grad

#### 4.3 Descenso por el Gradiente

La regla de actualización en cada iteración es:

$$\theta := \theta - \frac{\alpha}{m} X^T (g(X\theta) - y)$$

Dado el alto número de features (n = 12 288), se elige una tasa $\alpha$ moderada y se registra el historial de costo para verificar la convergencia.

In [ ]:
def descensoGradiente(theta, X, y, alpha, num_iters):
    """
    Descenso por el gradiente para regresión logística.
    Retorna (theta_optimizado, historial_costo).
    """
    m = y.shape[0]
    theta = theta.copy()
    J_history = []
    for it in range(num_iters):
        h = sigmoid(X.dot(theta))
        theta -= (alpha / m) * X.T.dot(h - y)
        J, _ = costFunction(theta, X, y)
        J_history.append(J)
        if (it + 1) % 50 == 0:
            print(f'  Iteración {it+1:4d}/{num_iters} — Costo: {J:.4f}')
    return theta, J_history

### 5. Entrenamiento OvR — 10 Clasificadores

Se entrena un clasificador binario independiente para cada una de las 10 clases de animales. Para la clase $k$, la etiqueta binaria $y_k^{(i)} = 1$ si el ejemplo $i$ es de esa especie, y $0$ en caso contrario.

> **Nota:** Con n = 12 288, cada iteración implica multiplicaciones matriciales de gran escala. Se recomienda usar una GPU en Colab (Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU).

In [ ]:
NUM_CLASES = len(ETIQUETAS)
alpha      = 0.05
num_iters  = 300
n_params   = X_train_b.shape[1]

all_theta     = np.zeros((NUM_CLASES, n_params))
all_J_history = []

for k in range(NUM_CLASES):
    print(f'\n[Clase {k}] Entrenando: {ETIQUETAS[k]}')
    y_k = (y_train == k).astype(float)
    theta_k = np.zeros(n_params)
    theta_k, J_hist_k = descensoGradiente(theta_k, X_train_b, y_k, alpha, num_iters)
    all_theta[k]  = theta_k
    all_J_history.append(J_hist_k)
    print(f'  → Costo final: {J_hist_k[-1]:.4f}')

print('\n✓ Entrenamiento completado.')

### 6. Gráficas de Costo

Se grafica la curva de convergencia del costo $J$ en función de las iteraciones para cada uno de los 10 clasificadores. Una curva descendente y estable indica buena convergencia.

In [ ]:
fig, axes = pyplot.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for k in range(NUM_CLASES):
    axes[k].plot(np.arange(num_iters), all_J_history[k], lw=2, color='darkorange')
    axes[k].set_title(f'Clase {k}: {ETIQUETAS[k]}', fontsize=9)
    axes[k].set_xlabel('Iteraciones', fontsize=8)
    axes[k].set_ylabel('Costo J', fontsize=8)
    axes[k].grid(True, alpha=0.3)

pyplot.suptitle('Curvas de Convergencia del Costo — OvR Animals-10', fontsize=14, fontweight='bold')
pyplot.tight_layout()
pyplot.show()

### 7. Predicción y Evaluación

#### 7.1 Función de predicción OvR

Para cada ejemplo se evalúan los 10 clasificadores y se selecciona la clase con la mayor probabilidad:

$$\hat{y} = \arg\max_{k \in \{0,...,9\}} \ h_{\theta_k}(x)$$

In [ ]:
def predecir_ovr(all_theta, X):
    """
    Predicción OvR — retorna índice de la clase con mayor probabilidad.
    all_theta: (K x n+1), X: (m x n+1)
    """
    probs = sigmoid(X.dot(all_theta.T))  # (m x K)
    return np.argmax(probs, axis=1)

In [ ]:
# Precisión en entrenamiento
y_pred_train = predecir_ovr(all_theta, X_train_b)
acc_train = np.mean(y_pred_train == y_train) * 100
print(f'Precisión en ENTRENAMIENTO: {acc_train:.2f}%')

# Precisión en prueba
y_pred_test = predecir_ovr(all_theta, X_test_b)
acc_test = np.mean(y_pred_test == y_test) * 100
print(f'Precisión en PRUEBA       : {acc_test:.2f}%')

#### 7.2 Matriz de Confusión

La diagonal muestra las predicciones correctas. Las celdas fuera de la diagonal indican confusiones entre clases — por ejemplo, el modelo podría confundir perros con gatos más frecuentemente que con arañas.

In [ ]:
def confusion_matrix_manual(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    return cm

cm = confusion_matrix_manual(y_test, y_pred_test, NUM_CLASES)

fig, ax = pyplot.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation='nearest', cmap='Oranges')
pyplot.colorbar(im, ax=ax)

tick_marks = np.arange(NUM_CLASES)
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(ETIQUETAS, rotation=45, ha='right')
ax.set_yticklabels(ETIQUETAS)

thresh = cm.max() / 2.0
for i in range(NUM_CLASES):
    for j in range(NUM_CLASES):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > thresh else 'black', fontsize=8)

ax.set_ylabel('Clase Real')
ax.set_xlabel('Clase Predicha')
ax.set_title('Matriz de Confusión — Conjunto de Prueba')
pyplot.tight_layout()
pyplot.show()

#### 7.3 Precisión por Clase

In [ ]:
precisiones_por_clase = []
print('Precisión por clase en el conjunto de prueba:')
for k in range(NUM_CLASES):
    mask_k = y_test == k
    acc_k = np.mean(y_pred_test[mask_k] == k) * 100 if mask_k.sum() > 0 else 0.0
    precisiones_por_clase.append(acc_k)
    print(f'  {ETIQUETAS[k]:10s}: {acc_k:.2f}%')

pyplot.figure(figsize=(11, 5))
bars = pyplot.bar(ETIQUETAS, precisiones_por_clase, color='darkorange', edgecolor='black')
pyplot.axhline(acc_test, color='blue', linestyle='--', label=f'Precisión global: {acc_test:.2f}%')
pyplot.xlabel('Categoría de Animal')
pyplot.ylabel('Precisión (%)')
pyplot.title('Precisión por Clase — Conjunto de Prueba (Animals-10)')
pyplot.xticks(rotation=30, ha='right')
pyplot.ylim(0, 115)
pyplot.legend()
for bar, val in zip(bars, precisiones_por_clase):
    pyplot.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
               f'{val:.1f}%', ha='center', va='bottom', fontsize=8)
pyplot.tight_layout()
pyplot.show()

#### 7.4 Predicciones Visuales

Se muestran 10 imágenes del conjunto de prueba con su clase real y la clase predicha por el modelo. Esta visualización es especialmente útil en datasets de imágenes para validar intuitivamente las predicciones.

In [ ]:
np.random.seed(42)
indices_muestra = np.random.choice(len(y_test), 10, replace=False)

fig, axes = pyplot.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

for ax, idx in zip(axes, indices_muestra):
    img = X_test[idx].reshape(IMG_SIZE, IMG_SIZE, 3)
    real = ETIQUETAS[y_test[idx]]
    pred = ETIQUETAS[y_pred_test[idx]]
    correcto = (y_test[idx] == y_pred_test[idx])
    color = 'green' if correcto else 'red'
    ax.imshow(img)
    ax.set_title(f'Real: {real}\nPred: {pred}', fontsize=8, color=color)
    ax.axis('off')

pyplot.suptitle('Predicciones en ejemplos del conjunto de prueba\n(Verde=correcto, Rojo=incorrecto)',
               fontsize=12, fontweight='bold')
pyplot.tight_layout()
pyplot.show()

### 8. Conclusiones

- El dataset **Animals-10** es multimodal: las imágenes RGB se aplanaron en vectores de **12 288 features**, cumpliendo ampliamente el requisito n > 100.
- Se cargaron y balancearon las imágenes con Pandas, garantizando el mismo número de ejemplos por clase.
- La normalización por división entre 255 lleva los píxeles al rango [0,1], lo que mejora la estabilidad numérica del descenso por gradiente.
- El esquema **One-vs-Rest** permite aplicar regresión logística binaria a un problema de 10 clases sin modificaciones al algoritmo central.
- Las curvas de costo muestran convergencia para todos los clasificadores dentro de las 300 iteraciones con $\alpha = 0.05$.
- La **visualización de predicciones** en imágenes reales valida intuitivamente la efectividad del modelo y permite identificar las confusiones más frecuentes.
- La precisión en prueba refleja la capacidad de generalización a datos nunca vistos durante el entrenamiento (20% reservado estrictamente).

**Repositorio GitHub:** https://github.com/jogaesmo/Escobar_Moscoso_Jorge_Gabriel_IA_1